In [1]:
import os
import warnings
import logging
from transformers import logging as hf_logging

# Stop bertscore's warnings
# 1. Set environment variables, use lowercase 'error'
os.environ["TRANSFORMERS_VERBOSITY"] = "error" # Trying 'error' to ensure it's parsed correctly
os.environ["TOKENIZERS_PARALLELISM"] = "false" # Suppress tokenizers-related warnings

# 2. Suppress all Python-level warnings (UserWarning)
warnings.filterwarnings("ignore") # This catches and ignores warnings from internal libraries like bert-score

# 3. Explicitly set transformers logging level (using a compatible function)
# set_verbosity_error() suppresses WARNING and INFO levels
hf_logging.set_verbosity_error()

# 4. Ensure PyTorch loggers are also suppressed
# Setting to CRITICAL is highly effective
logging.getLogger("torch").setLevel(logging.CRITICAL)
logging.getLogger("pytorch_lightning").setLevel(logging.CRITICAL)

# 5. Ensure the root logger level is high enough
logging.getLogger().setLevel(logging.CRITICAL)

In [2]:
!pip install bert-score
from bert_score import score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 5.5 MB/s eta 0:00:00


In [3]:
# load the files
from google.colab import drive
drive.mount('/content/drive')

# load the QAs
import json
manual_QAs_json_path = "/content/drive/MyDrive/Haystack/Dataset/manual_QAs.json"
with open(manual_QAs_json_path, "r", encoding="utf-8") as qas_file:
  manual_QAs = json.load(qas_file)

# load the manual
file_path = "/content/drive/MyDrive/Haystack/Dataset/manual.json"
import json
with open(file_path, "r", encoding="utf-8") as file:
  manual=json.load(file)
# save the contexts, titles and pages for later use
contexts = [document["context"] for document in manual]
titles = [document["title"] for document in manual]
pages = [document["page"] for document in manual]

# load the results
names = ["last", "all", "rewrite"]
manual_results = {}
manual_evaluations = {}
for name1 in names:
  for name2 in names:
    json_path = f"""/content/drive/MyDrive/Haystack/Result/important_results/result_{name1}_{name2}.json"""
    with open(json_path, "r", encoding="utf-8") as file:
      manual_results[f"{name1}_{name2}"] = json.load(file) # Fixed: Changed json_path to file
      manual_evaluations[f"{name1}_{name2}"] = {}

Mounted at /content/drive


In [4]:
# manual_results
print(manual_results.keys())
print(manual_results['last_last'].keys())
print(manual_results['last_last']["LDS"].keys())
print(manual_results['last_last']["LDS"]["top_k=3"][0].keys())

dict_keys(['last_last', 'last_all', 'last_rewrite', 'all_last', 'all_all', 'all_rewrite', 'rewrite_last', 'rewrite_all', 'rewrite_rewrite'])
dict_keys(['LDS', 'ESS', 'RNS'])
dict_keys(['top_k=1', 'top_k=3', 'top_k=5', 'top_k=10'])
dict_keys(['Responses', 'Retrieved_context_ids'])


In [5]:
# manual_QAs
print(manual_QAs.keys())
print(manual_QAs['LDS'][0].keys())

dict_keys(['LDS', 'ESS', 'RNS'])
dict_keys(['Queries', 'Answers', 'Context_ids'])


In [6]:
# functions for evaluating one conversation
def retrieval_evaluation(retrieval_type, response_type, qas_type, top_k, sequence):
  # load the correct_context_ids and retrieved_context_ids
  correct_context_id = manual_QAs[qas_type][sequence]['Context_ids']
  correct_context_ids = [[id for _ in range(top_k)] for id in correct_context_id]
  retrieved_context_ids = manual_results[f"{retrieval_type}_{response_type}"][qas_type][f"top_k={top_k}"][sequence]["Retrieved_context_ids"]

  # convert the ids into contexts
  correct_contexts = [[contexts[id] for id in one_query_ids] for one_query_ids in correct_context_ids]
  retrieved_contexts = [[contexts[id] for id in one_query_ids] for one_query_ids in retrieved_context_ids]

  # calculate the bertscores
  bertscores = {"Ps": [], "Rs": [], "F1s": []}
  for i in range(len(correct_contexts)):
    P, R, F1 = score(retrieved_contexts[i], correct_contexts[i], lang="en", verbose = False, device = "cuda")
    bertscores["Ps"].append(P)
    bertscores["Rs"].append(R)
    bertscores["F1s"].append(F1)

  return bertscores

def response_evaluation(retrieval_type, response_type, qas_type, top_k, sequence):
  # load the answers and responses
  correct_answers = manual_QAs[qas_type][sequence]['Answers']
  llm_responses = manual_results[f"{retrieval_type}_{response_type}"][qas_type][f"top_k={top_k}"][sequence]["Responses"]

  #c  calculate the bertscores
  P, R, F1 = score(llm_responses, correct_answers, lang="en", verbose = False, device = "cuda")
  return {"Ps": P, "Rs": R, "F1s": F1}

In [7]:
#retrieval_evaluation(retrieval_type = "last", response_type = "last", qas_type = "LDS", top_k = 3, sequence = 0)
#response_evaluation(retrieval_type = "last", response_type = "last", qas_type = "LDS", top_k = 3, sequence = 0)

In [8]:
# initialize manual_evaluations
names = ["last", "all", "rewrite"]
qas_types = ["LDS", "ESS", "RNS"]
for retrieval_type in names:
  for response_type in names:
    for qas_type in qas_types:
      manual_evaluations[f"{retrieval_type}_{response_type}"][qas_type] = {"top_k=1": None, "top_k=3": None, "top_k=5": None, "top_k=10": None}

In [9]:
# functions for evaluation many conversations
def evaluation(retrieval_type, response_type, qas_type, top_k):
  evaluation_result = {"retrieval": [], "response": []}
  for i in range(40):
    if (i%5 == 0):
      print(i+1, end = " ")
    if manual_results[f"{retrieval_type}_{response_type}"][qas_type][f"top_k={top_k}"][i] == "Error": # skip errors
      evaluation_result['retrieval'].append("Error")
      evaluation_result['response'].append("Error")
    else:
      evaluation_result['retrieval'].append(retrieval_evaluation(retrieval_type = retrieval_type, response_type = response_type, qas_type = qas_type, top_k = top_k, sequence = i))
      evaluation_result['response'].append(response_evaluation(retrieval_type = retrieval_type, response_type = response_type, qas_type = qas_type, top_k = top_k, sequence = i))
  return evaluation_result

def dump_evaluation(manual_evaluations, retrieval_type, response_type, qas_type, top_k):
  manual_evaluations[f"{retrieval_type}_{response_type}"][qas_type][f"top_k={top_k}"] = evaluation(retrieval_type, response_type, qas_type, top_k)

In [10]:
# codes for saved process

# last_last, last_all, last_rewrite
# all_last, all_all, all_rewrite
# rewrite_last, rewrite_all, rewrite_rewrite
# LDS, ESS, RNS
# top_k: 1 3 5 10

# ex) last_last_LDS_top_k=1
# ex) last_last_LDS_topk=10

# Helper function to convert Tensors to JSON serializable types
import torch

def convert_tensors_to_serializable(obj):
    if isinstance(obj, torch.Tensor):
        # For scalar tensors, use .item(). For multi-dimensional, use .tolist()
        if obj.ndim == 0:
            return obj.item()
        else:
            return obj.tolist()
    elif isinstance(obj, dict):
        return {k: convert_tensors_to_serializable(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [convert_tensors_to_serializable(elem) for elem in obj]
    else:
        return obj

In [11]:
# dump_evaluation(manual_evaluations, retrieval_type = "last", response_type = "last", qas_type="LDS", top_k=1)

# # save the process
# json_path = "/content/drive/MyDrive/Haystack/Result/evaluations/last_last_LDS_top_k=1.json"

# # Convert the manual_evaluations dictionary to a JSON-serializable format
# serializable_evaluations = convert_tensors_to_serializable(manual_evaluations)

# with open(json_path, 'w') as f:
#   json.dump(serializable_evaluations, f, indent=4)

In [12]:
# json_path = "/content/drive/MyDrive/Haystack/Result/evaluations/last_last_LDS_top_k=1.json"
# with open(json_path, "r", encoding="utf-8") as file:
#   manual_evaluations = json.load(file)

# dump_evaluation(manual_evaluations, retrieval_type = "last", response_type = "last", qas_type="LDS", top_k=3)
# dump_evaluation(manual_evaluations, retrieval_type = "last", response_type = "last", qas_type="LDS", top_k=5)
# dump_evaluation(manual_evaluations, retrieval_type = "last", response_type = "last", qas_type="LDS", top_k=10)

# json_path = "/content/drive/MyDrive/Haystack/Result/evaluations/last_last_LDS_top_k=10.json"
# # Convert the manual_evaluations dictionary to a JSON-serializable format before saving
# serializable_evaluations_updated = convert_tensors_to_serializable(manual_evaluations)
# with open(json_path, 'w') as f:
#   json.dump(serializable_evaluations_updated, f, indent=4)

In [13]:
def huge_evaluation(retrieval_type, response_type, qas_type):
  dump_evaluation(manual_evaluations, retrieval_type, response_type, qas_type, top_k=1)
  dump_evaluation(manual_evaluations, retrieval_type, response_type, qas_type, top_k=3)
  dump_evaluation(manual_evaluations, retrieval_type, response_type, qas_type, top_k=5)
  dump_evaluation(manual_evaluations, retrieval_type, response_type, qas_type, top_k=10)

  json_path = f"/content/drive/MyDrive/Haystack/Result/evaluations/{retrieval_type}_{response_type}_{qas_type}_top_k=10.json"
  serializable_evaluations_updated = convert_tensors_to_serializable(manual_evaluations)
  with open(json_path, 'w') as f:
    json.dump(serializable_evaluations_updated, f, indent=4)

In [14]:
# # last retrieval
# huge_evaluation("last", "last", "ESS")
# huge_evaluation("last", "last", "RNS")

# huge_evaluation("last", "all", "LDS")
# huge_evaluation("last", "all", "ESS")
# huge_evaluation("last", "all", "RNS")

# huge_evaluation("last", "rewrite", "LDS")
# huge_evaluation("last", "rewrite", "ESS")
# huge_evaluation("last", "rewrite", "RNS")

In [15]:
# huge_evaluation("last", "last", "RNS")

# huge_evaluation("last", "all", "LDS")
# huge_evaluation("last", "all", "ESS")
# huge_evaluation("last", "all", "RNS")

# huge_evaluation("last", "rewrite", "LDS")
# huge_evaluation("last", "rewrite", "ESS")
# huge_evaluation("last", "rewrite", "RNS")

In [16]:
# # all retrieval
# huge_evaluation("all", "last", "LDS")
# huge_evaluation("all", "last", "ESS")
# huge_evaluation("all", "last", "RNS")

# huge_evaluation("all", "all", "LDS")
# huge_evaluation("all", "all", "ESS")
# huge_evaluation("all", "all", "RNS")

# huge_evaluation("all", "rewrite", "LDS")
# huge_evaluation("all", "rewrite", "ESS")
# huge_evaluation("all", "rewrite", "RNS")

In [17]:
json_path = "/content/drive/MyDrive/Haystack/Result/evaluations/all_rewrite_RNS_top_k=10.json"
with open(json_path, "r", encoding="utf-8") as file:
  manual_evaluations = json.load(file)

In [18]:
# show the progress
for retrieval_type in names:
  print("retrieval_type: ", retrieval_type, end = "\t")
  for response_type in names:
    print("response_type: ", response_type)
    for qas_type in qas_types:
      print("\t", "qas_type: ", qas_type)
      for top_k in [1, 3, 5, 10]:
        print("\t\t", f"top_k={top_k}", end = " ")
        if manual_evaluations[f"{retrieval_type}_{response_type}"][qas_type][f"top_k={top_k}"] == None:
          print("retrieval: 0", end = " ")
          print("response: 0", end = " ")
        else:
          print("retrieval: ", len(manual_evaluations[f"{retrieval_type}_{response_type}"][qas_type][f"top_k={top_k}"]["retrieval"]), end = " ")
          print("response: ", len(manual_evaluations[f"{retrieval_type}_{response_type}"][qas_type][f"top_k={top_k}"]["response"]), end = " ")
        print()

retrieval_type:  last	response_type:  last
	 qas_type:  LDS
		 top_k=1 retrieval:  40 response:  40 
		 top_k=3 retrieval:  40 response:  40 
		 top_k=5 retrieval:  40 response:  40 
		 top_k=10 retrieval:  40 response:  40 
	 qas_type:  ESS
		 top_k=1 retrieval:  40 response:  40 
		 top_k=3 retrieval:  40 response:  40 
		 top_k=5 retrieval:  40 response:  40 
		 top_k=10 retrieval:  40 response:  40 
	 qas_type:  RNS
		 top_k=1 retrieval:  40 response:  40 
		 top_k=3 retrieval:  40 response:  40 
		 top_k=5 retrieval:  40 response:  40 
		 top_k=10 retrieval:  40 response:  40 
response_type:  all
	 qas_type:  LDS
		 top_k=1 retrieval:  40 response:  40 
		 top_k=3 retrieval:  40 response:  40 
		 top_k=5 retrieval:  40 response:  40 
		 top_k=10 retrieval:  40 response:  40 
	 qas_type:  ESS
		 top_k=1 retrieval:  40 response:  40 
		 top_k=3 retrieval:  40 response:  40 
		 top_k=5 retrieval:  40 response:  40 
		 top_k=10 retrieval:  40 response:  40 
	 qas_type:  RNS
		 top_k=1

In [19]:
huge_evaluation("all", "last", "LDS")

1 

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

6 11 16 21 26 31 36 1 6 11 16 21 26 31 36 1 6 11 16 21 26 31 36 1 6 11 16 21 26 31 36 

In [20]:
# rewrite retrieval
huge_evaluation("rewrite", "last", "LDS")
huge_evaluation("rewrite", "last", "ESS")
huge_evaluation("rewrite", "last", "RNS")

huge_evaluation("rewrite", "all", "LDS")
huge_evaluation("rewrite", "all", "ESS")
huge_evaluation("rewrite", "all", "RNS")

huge_evaluation("rewrite", "rewrite", "LDS")
huge_evaluation("rewrite", "rewrite", "ESS")
huge_evaluation("rewrite", "rewrite", "RNS")

1 6 11 16 21 26 31 36 1 6 11 16 21 26 31 36 1 6 11 16 21 26 31 36 1 6 11 16 21 26 31 36 1 6 11 16 21 26 31 36 1 6 11 16 21 26 31 36 1 6 11 16 21 26 31 36 1 6 11 16 21 26 31 36 1 6 11 16 21 26 31 36 1 6 11 16 21 26 31 36 1 6 11 16 21 26 31 36 1 6 11 16 21 26 31 36 1 6 11 16 21 26 31 36 1 6 11 16 21 26 31 36 1 6 11 16 21 26 31 36 1 6 11 16 21 26 31 36 1 6 11 16 21 26 31 

HTTP Error 504 thrown while requesting HEAD https://huggingface.co/roberta-large/resolve/main/config.json
Retrying in 1s [Retry 1/5].


36 1 6 11 16 21 26 31 36 1 6 11 16 21 26 31 36 1 6 11 16 21 26 31 36 1 6 11 16 21 26 31 36 1 6 11 16 21 26 31 36 1 6 11 16 21 26 31 36 1 6 11 16 21 26 31 36 1 6 

HTTP Error 504 thrown while requesting HEAD https://huggingface.co/roberta-large/resolve/main/tokenizer_config.json
Retrying in 1s [Retry 1/5].


11 16 21 26 31 36 1 6 11 16 21 26 31 36 1 6 11 16 21 26 31 36 1 6 11 16 21 26 31 36 1 6 11 16 21 26 31 36 1 6 11 16 21 26 31 36 1 6 11 16 21 26 31 36 1 6 11 16 21 26 31 36 1 6 11 16 21 26 31 36 1 6 11 16 21 26 31 36 1 6 11 16 21 26 31 36 1 6 11 16 21 26 31 36 